In [31]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


# 1. Load dataset

In [32]:
TS_FILE = "Dataset/Building_AllPoints_15m_2025Jan01_2025Oct01_COV_and_15min.csv"

ts_df = pd.read_csv(TS_FILE)

# timestamp column
TIME_COL = "Timestamp"
if TIME_COL in ts_df.columns:
    ts_df[TIME_COL] = pd.to_datetime(ts_df[TIME_COL])
    ts_df = ts_df.set_index(TIME_COL)

print("original column number:", len(ts_df.columns))
ts_df.head()

original column number: 15


,AHU-2 AvgCcoilTmp,AHU-2 AvgMaTmp,AHU-2 ChwEnTmp,AHU-2 ChwVlvPos,AHU-2 MaTmp1,AHU-2 OaFl,AHU-2 OaTmp,AHU-2 SaFanASts,AHU-2 SaFanBSts,AHU-2 SaFanCSts,AHU-2 SaFanDSts,AHU-2 SaFanESts,AHU-2 SaFanFSts,AHU-2 SaFl,AHU-2 SaTmp
Timestamp,,,,,,,,,,,,,,,
2025-01-01 00:00:00,59.415386,66.289879,35.0,7.624306,66.010521,146.25,53.130001,1.0,1.0,1.0,1.0,1.0,1.0,14376.082031,61.399986
2025-01-01 00:15:00,58.561531,66.150818,35.0,0.000000,65.963142,146.25,52.810001,1.0,1.0,1.0,1.0,1.0,1.0,14283.847656,59.600025
2025-01-01 00:30:00,62.046177,66.010521,35.0,5.511208,65.868423,130.00,52.055149,1.0,1.0,1.0,1.0,1.0,1.0,13910.714844,62.323051
2025-01-01 00:45:00,57.015396,65.915802,35.0,1.837433,65.678955,6266.00,51.779999,1.0,1.0,1.0,1.0,1.0,1.0,13411.806641,59.369274
2025-01-01 01:00:00,59.069244,65.702621,35.0,0.018471,65.536819,139.75,51.709999,1.0,1.0,1.0,1.0,1.0,1.0,13910.714844,59.923100


# 2. Name-based auto labeling

In [33]:
def auto_label_from_name(var_name: str) -> dict:
    name = var_name.lower()

    tag_temperature = 0
    tag_flow = 0
    tag_pressure = 0
    tag_status = 0
    tag_command = 0
    tag_mode = 0

    # temperature
    if ("tmp" in name) or ("temp" in name):
        tag_temperature = 1

    # flow
    if ("fl" in name) or ("flow" in name):
        tag_flow = 1

    # pressure
    if ("ps" in name) or ("press" in name):
        tag_pressure = 1

    # status
    if ("sts" in name) or ("status" in name) or ("st" in name and "fan" in name):
        tag_status = 1

    # command
    if ("cmd" in name) or ("command" in name):
        tag_command = 1

    # mode
    if ("mode" in name):
        tag_mode = 1

    return {
        "tag_temperature": tag_temperature,
        "tag_flow": tag_flow,
        "tag_pressure": tag_pressure,
        "tag_status": tag_status,
        "tag_command": tag_command,
        "tag_mode": tag_mode,
    }

TAG_COLS = [
    "tag_temperature",
    "tag_flow",
    "tag_pressure",
    "tag_status",
    "tag_command",
    "tag_mode",
]

# 3. Generate time-series label

In [34]:
value_cols = ts_df.columns.tolist()
print("Time-series variable number:", len(value_cols))

X_list = []
y_list = []
var_names = []

for col in value_cols:
    series = pd.to_numeric(ts_df[col], errors="coerce")

    # NaN -> exclude 
    if series.isna().all():
        continue

    # conservate 
    series = series.fillna(method="ffill").fillna(method="bfill")

    # z-score regularization
    std = series.std()
    if std == 0 or np.isnan(std):
        norm_series = series.values.astype("float32")
    else:
        norm_series = ((series - series.mean()) / (std + 1e-6)).values.astype("float32")

    X_list.append(norm_series)
    var_names.append(col)

    tags = auto_label_from_name(col)
    y_list.append([tags[k] for k in TAG_COLS])

print("effective variable number:", len(X_list))

Time-series variable number: 15
유효 변수 개수: 15


/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_73749/4209147734.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method="ffill").fillna(method="bfill")


# 4. Sequence Length

In [35]:
seq_lengths = [len(x) for x in X_list]
min_len = min(seq_lengths)
max_len = max(seq_lengths)
print("min sequence length:", min_len, "/ max length:", max_len)

TARGET_LEN = min_len

X_arr = []
for seq in X_list:
    if len(seq) > TARGET_LEN:
        X_arr.append(seq[-TARGET_LEN:])  
    else:
        X_arr.append(seq)  

X_arr = np.stack(X_arr, axis=0) 
y_arr = np.array(y_list, dtype="float32")

print("X_arr shape:", X_arr.shape) 
print("y_arr shape:", y_arr.shape) 

min sequence length: 26209 / max length: 26209
X_arr shape: (15, 26209)
y_arr shape: (15, 6)


# 5. Split train/test

In [36]:
X_arr = X_arr[..., np.newaxis]
print("LSTM input shape:", X_arr.shape)

X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(
    X_arr, y_arr, var_names, test_size=0.3, random_state=42
)

print("Train samples:", X_train.shape[0])
print("Test samples :", X_test.shape[0])

LSTM input shape: (15, 26209, 1)
Train samples: 10
Test samples : 5


# 6. define and train LSTM model

In [37]:
model = Sequential([
    LSTM(32, input_shape=(X_train.shape[1], 1)),
    Dense(32, activation="relu"),
    Dense(len(TAG_COLS), activation="sigmoid")  # multi-label
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=8,
    callbacks=[early_stop],
    verbose=1,
)

Epoch 1/100


/opt/anaconda3/envs/ncphd/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6916 - val_accuracy: 0.0000e+00 - val_loss: 0.6909
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6882 - val_accuracy: 0.0000e+00 - val_loss: 0.6891
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6851 - val_accuracy: 0.0000e+00 - val_loss: 0.6873
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6821 - val_accuracy: 0.0000e+00 - val_loss: 0.6855
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6791 - val_accuracy: 0.0000e+00 - val_loss: 0.6836
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6250 - loss: 0.6761 - val_accuracy: 0.0000e+00 - val_loss: 0.6818
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6250 - loss: 0.6731 - val_accuracy: 0.0000e+00 - val_loss: 0.6799
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6250 - loss: 0.6701 - val_accuracy: 0.0000e+00 - val_l

# 7. Evaluate performance

In [39]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

micro_prec = precision_score(y_test, y_pred, average="micro", zero_division=0)
micro_rec  = recall_score(y_test, y_pred, average="micro", zero_division=0)
micro_f1   = f1_score(y_test, y_pred, average="micro", zero_division=0)

macro_prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1   = f1_score(y_test, y_pred, average="macro", zero_division=0)

hloss = hamming_loss(y_test, y_pred)

print("\n####### LSTM Baseline (Name-based Tag) #######")
print(f"  - Micro Precision: {micro_prec:.3f}")
print(f"  - Micro Recall   : {micro_rec:.3f}")
print(f"  - Micro F1       : {micro_f1:.3f}")
print(f"  - Macro Precision: {macro_prec:.3f}")
print(f"  - Macro Recall   : {macro_rec:.3f}")
print(f"  - Macro F1       : {macro_f1:.3f}")
print(f"  - Hamming Loss   : {hloss:.3f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step

####### LSTM Baseline (Name-based Tag) #######
  - Micro Precision: 0.600
  - Micro Recall   : 0.600
  - Micro F1       : 0.600
  - Macro Precision: 0.222
  - Macro Recall   : 0.333
  - Macro F1       : 0.250
  - Hamming Loss   : 0.133


In [40]:
for i, tag in enumerate(TAG_COLS):
    f1 = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    print(f"{tag}: F1 = {f1:.3f}")

tag_temperature: F1 = 0.500
tag_flow: F1 = 0.000
tag_pressure: F1 = 0.000
tag_status: F1 = 1.000
tag_command: F1 = 0.000
tag_mode: F1 = 0.000


In [44]:
subset_acc = (y_pred == y_test).all(axis=1).mean()
print("Accuracy:", subset_acc)

results = pd.DataFrame({
    "Variable": names_test
})

for i, tag in enumerate(TAG_COLS):
    results[tag] = y_pred[:, i]

results

Accuracy: 0.6


,Variable,tag_temperature,tag_flow,tag_pressure,tag_status,tag_command,tag_mode
0,AHU-2 SaFanCSts,0,0,0,1,0,0
1,AHU-2 SaFanESts,0,0,0,1,0,0
2,AHU-2 AvgCcoilTmp,1,0,0,0,0,0
3,AHU-2 SaFl,1,0,0,0,0,0
4,AHU-2 OaFl,1,0,0,0,0,0
